In [11]:
from google.colab import files
import cv2
import matplotlib.pyplot as plt

# upload image
uploaded = files.upload()

# get uploaded file name
image_path = next(iter(uploaded))

# analyze image
result = analyze_crop(image_path, model)

# display image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)
plt.axis('off')
plt.show()

# final decision
x = final_decision(result)

Saving apple.jpg to apple (2).jpg


NameError: name 'analyze_crop' is not defined

In [ ]:
def analyze_crop(image_path, model):
    results = {}

    # classification
    crop, conf = predict_fruit(image_path, model)
    results["crop"] = crop
    results["confidence"] = conf

    # remove background
    no_bg_image = remove_background(image_path)

    # mask for texture
    mask = get_mask(no_bg_image)

    # methods
    methods_config = crop_methods.get(crop, [])

    analysis_results = {}

    for item in methods_config:
        method = item["method"]
        params = item["params"]

        # texture analysis needs image + mask
        if method.__name__ in ["texture_analysis", "color_analysis", "sprouting_detection"]:
            output = method(
                crop=np.array(no_bg_image.convert("RGB")),
                mask=mask,
                **params
            )
        else:
          img = np.array(no_bg_image.convert("RGBA"))
          output = method(
              img,
              mask=mask,
              **params
              )

        analysis_results[method.__name__] = output

    results["analysis"] = analysis_results

    return results

In [ ]:
def final_decision(result):

    '''
    Excellent:
    No defects detected. Fresh crop with high quality.

    Acceptable:
    One minor defect detected (either spot, texture, or non-uniform color).
    Crop is still eatable.

    Poor:
    Multiple defects or serious defect detected (mold or sprouting).
    Crop is low quality and not eatable.
    '''

    crop_name = result["crop"]
    confidence = result["confidence"]
    analysis = result["analysis"]

    issues = []
    spot_percentage = 0
    mold_detected = False
    sprouted = False

    if "detect_spots" in analysis:

        spot_result = analysis["detect_spots"]

        if isinstance(spot_result, str):

            if "mold" in spot_result.lower():
                mold_detected = True
                issues.append("Mold")

            elif "dark" in spot_result.lower():
                issues.append("Dark Spots")

            try:
                spot_percentage = float(
                    spot_result.split(",")[1].replace("%", "").strip()
                )
            except:
                spot_percentage = 0

    if "texture_analysis" in analysis:

        texture = analysis["texture_analysis"] or {}

        if texture.get("is_shriveled") or texture.get("has_texture_problem"):
            issues.append("Texture Problem")

    if "color_analysis" in analysis:

        color = analysis["color_analysis"] or {}

        if not color.get("passed", True):
            issues.append("Non-uniform Color")

    if "sprout_analysis" in analysis:

        sprout = analysis["sprout_analysis"] or {}

        if sprout.get("sprouted", False):
            sprouted = True
            issues.append("Sprouting")

    if crop_name.lower() == "banana":

        if mold_detected:
            quality = "Poor"

        else:

            other_issues = [
                i for i in issues
                if i not in ["Mold", "Dark Spots"]
            ]

            if len(other_issues) >= 2:
                quality = "Poor"

            else:

                if 0 <= spot_percentage <= 19:
                    quality = "Excellent"

                elif 19 < spot_percentage <= 50:
                    quality = "Acceptable"

                else:
                    quality = "Poor"

    else:

        minor_issues = [
            i for i in issues
            if i in ["Dark Spots", "Texture Problem", "Non-uniform Color"]
        ]

        if mold_detected or sprouted:
            quality = "Poor"

        elif len(minor_issues) == 0:
            quality = "Excellent"

        elif len(minor_issues) == 1:
            quality = "Acceptable"

        else:
            quality = "Poor"

    print("=" * 13)
    print("     Crop Analysis Result")
    print("=" * 13)

    print(f"\nCrop: {crop_name}")
    print(f"Confidence: {confidence:.2%}")

    print(f"\nQuality: {quality}")

    print("\nDetected Issues:")

    if len(issues) == 0:
        print("• None")

    else:
        for issue in issues:
            print(f"• {issue}")

    if crop_name.lower() == "banana":
        print(f"\nSpot Percentage: {spot_percentage:.2f}%")

    return quality

# **1 - Classification**

In [3]:
pip install ultralytics

In [4]:
from ultralytics import YOLO
from google.colab import drive

model = YOLO("/content/drive/MyDrive/GP-2/YOLOv8l-cls_RAND/weights/best.pt")

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def predict_fruit(image_path, model):
    class_names = {
        0: 'Apple', 1: 'Avocado', 2: 'Banana', 3: 'Blackberry',
        4: 'Blueberry', 5: 'Carrot', 6: 'Cherry', 7: 'Cucumber',
        8: 'Eggplant', 9: 'Ginger', 10: 'Green Grape', 11: 'Green Pepper',
        12: 'Guava', 13: 'Kiwi', 14: 'Lemon', 15: 'Lime',
        16: 'Mango', 17: 'Orange', 18: 'Orange Pepper', 19: 'Papaya',
        20: 'Peach', 21: 'Pear', 22: 'Pomegranate', 23: 'Potato',
        24: 'Red Grape', 25: 'Red Onion', 26: 'Red Pepper',
        27: 'Strawberry', 28: 'Tomato', 29: 'White Onion',
        30: 'Yellow Pepper', 31: 'Zucchini'
    }

    results = model(image_path)
    probs = results[0].probs

    top1 = int(probs.top1)
    conf = float(probs.top1conf)

    pred_class = class_names[top1]

    return pred_class, conf

In [7]:
pip install rembg

In [8]:
pip install "rembg[gpu]"

  Using cached onnxruntime_gpu-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.6 kB)
Using cached onnxruntime_gpu-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (277.0 MB)


- Remove background

In [ ]:
from rembg import remove
from PIL import Image
import io

def remove_background(input_path):
    with open(input_path, "rb") as input_file:
        input_data = input_file.read()

    output_data = remove(input_data)

    output_image = Image.open(io.BytesIO(output_data)).convert("RGBA")

    return output_image

In [10]:
crop_methods = {

    "Apple": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 15,
                "reference_lab": [165, 135, 135],
                "reference_shift_threshold": 50
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 8,
                "skin_texture_threshold": 0.60
            }
        }
    ],

    "Avocado": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [30, 130, 120],
                "reference_shift_threshold": 18
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 50,
                "canny_high": 130,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 10,
                "skin_texture_threshold": 0.75
            }
        }
    ],

    "Banana": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 40,
                "reference_lab": [180, 130, 180],
                "reference_shift_threshold": 25
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 7,
                "canny_low": 20,
                "canny_high": 80,
                "wrinkle_kernel_size": 11,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.60
            }
        }
    ],

    "Blackberry": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [55, 20, 45],
                "delta_e_threshold": 60,
                "min_coverage": 0.65
            }
        }
    ],

    "Blueberry": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 5,
                "reference_lab": [40, 140, 130],
                "reference_shift_threshold": 10
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 25,
                "canny_high": 90,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 4,
                "skin_texture_threshold": 0.55
            }
        }
    ],

    "Carrot": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [20, 130, 240],
                "delta_e_threshold": 50,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [160, 145, 200],
                "reference_shift_threshold": 100
            }
        }
    ],

    "Cherry": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [15, 15, 130],
                "delta_e_threshold": 65,
                "min_coverage": 0.65
            }
        },
         {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 40,
                "canny_high": 120,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.78
                }
             },

        {
            "method": detect_spots,
            "params": {
                 "threshold": 15,
                "reference_lab": [77, 155, 135],
                "reference_shift_threshold": 10
                 }

            }
    ],

    "Cucumber": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [150, 125, 130],
                "reference_shift_threshold": 100
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 30,
                "canny_high": 100,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 15,
                "skin_texture_threshold": 0.62
            }
        }
    ],

    "Eggplant": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [45, 5, 40],
                "delta_e_threshold": 70,
                "min_coverage": 0.55
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 40,
                "reference_lab": [60, 135, 100],
                "reference_shift_threshold": 25
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 30,
                "canny_high": 100,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.60
            }
        }
    ],

    "Ginger": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [140, 135, 145],
                "reference_shift_threshold": 20
            }
        }
    ],

    "Green Grape": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [130, 190, 150],
                "delta_e_threshold": 60,
                "min_coverage": 0.70
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [170, 130, 145],
                "reference_shift_threshold": 15
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 25,
                "canny_high": 90,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 5,
                "skin_texture_threshold": 0.58
            }
        }
    ],

    "Green Pepper": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [15, 100, 25],
                "delta_e_threshold": 70,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [140, 125, 140],
                "reference_shift_threshold": 20
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.62
            }
        }
    ],

    "Guava": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [150, 135, 135],
                "reference_shift_threshold": 50
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 7,
                "skin_texture_threshold": 0.60
            }
        }
    ],

    "Kiwi": [
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 7,
                "canny_low": 50,
                "canny_high": 140,
                "wrinkle_kernel_size": 11,
                "shrivel_threshold": 15,
                "skin_texture_threshold": 0.80
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [110, 130, 120],
                "reference_shift_threshold": 100
            }
        }
    ],

    "Lemon": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [35, 190, 225],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [225, 120, 160],
                "reference_shift_threshold": 30
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 40,
                "canny_high": 120,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 7,
                "skin_texture_threshold": 0.68
            }
        }
    ],

    "Lime": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [45, 170, 100],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [180, 130, 140],
                "reference_shift_threshold": 30
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 40,
                "canny_high": 120,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 7,
                "skin_texture_threshold": 0.68
            }
        }
    ],

    "Mango": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 30,
                "reference_lab": [170, 125, 155],
                "reference_shift_threshold": 55
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.62
            }
        }
    ],

    "Orange": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 25,
                "reference_lab": [190, 140, 180],
                "reference_shift_threshold": 10
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 40,
                "canny_high": 120,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 9,
                "skin_texture_threshold": 0.72
            }
        },
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [30, 115, 210],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        }
    ],

    "Orange Pepper": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [35, 135, 245],
                "delta_e_threshold": 70,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [175, 140, 175],
                "reference_shift_threshold": 15
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.62
            }
        }
    ],

    "Papaya": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [205, 140, 165],
                "reference_shift_threshold": 20
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.60
            }
        }
    ],

    "Peach": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [190, 140, 150],
                "reference_shift_threshold": 50
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 25,
                "canny_high": 90,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 5,
                "skin_texture_threshold": 0.58
            }
        }
    ],

    "Pear": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [200, 130, 140],
                "reference_shift_threshold": 40
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 7,
                "skin_texture_threshold": 0.63
            }
        }
    ],

    "Pomegranate": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 20,
                "reference_lab": [80, 160, 130],
                "reference_shift_threshold": 30
            }
        },

          {"method": texture_analysis, "params": {
              "blur_kernel": 5,
              "canny_low": 30,
              "canny_high": 100,
              "wrinkle_kernel_size": 9,
              "shrivel_threshold": 5,
              "skin_texture_threshold": 0.55
    }
           }
    ],

    "Potato": [

        {
            "method": sprouting_detection,
            "params": {}
        }
    ],

    "Red Grape": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [45, 15, 105],
                "delta_e_threshold": 60,
                "min_coverage": 0.70
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 15,
                "reference_lab": [70, 140, 120],
                "reference_shift_threshold": 20
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 25,
                "canny_high": 90,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 5,
                "skin_texture_threshold": 0.58
            }
        }
    ],

    "Red Pepper": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [30, 30, 230],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [90, 150, 180],
                "reference_shift_threshold": 20
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 25,
                "canny_high": 90,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 5,
                "skin_texture_threshold": 0.58
            }
        }
    ],

    "Red Onion": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 15,
                "reference_lab": [90, 135, 140],
                "reference_shift_threshold": 20
            }
        },
        {
            "method": sprouting_detection,
            "params": {}
        }
    ],

    "Strawberry": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 30,
                "reference_lab": [80, 150, 140],
                "reference_shift_threshold": 15
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 30,
                "canny_high": 100,
                "wrinkle_kernel_size": 7,
                "shrivel_threshold": 8,
                "skin_texture_threshold": 0.65
            }
        }
    ],

    "Tomato": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [35, 50, 205],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 30,
                "canny_high": 100,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.45
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 30,
                "reference_lab": [110, 150, 170],
                "reference_shift_threshold": 100
            }
        }
    ],

    "White Onion": [
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [210, 125, 130],
                "reference_shift_threshold": 30
            }
        },
        {
            "method": sprouting_detection,
            "params": {}
        }
    ],

    "Zucchini": [
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.62
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 10,
                "reference_lab": [150, 125, 135],
                "reference_shift_threshold": 40
            }
        }
    ],

    "Yellow Pepper": [
        {
            "method": color_analysis,
            "params": {
                "ref_color_bgr": [35, 190, 225],
                "delta_e_threshold": 60,
                "min_coverage": 0.75
            }
        },
        {
            "method": detect_spots,
            "params": {
                "threshold": 30,
                "reference_lab": [210, 135, 170],
                "reference_shift_threshold": 30
            }
        },
        {
            "method": texture_analysis,
            "params": {
                "blur_kernel": 5,
                "canny_low": 35,
                "canny_high": 110,
                "wrinkle_kernel_size": 9,
                "shrivel_threshold": 6,
                "skin_texture_threshold": 0.62
            }
        }
    ]
}

NameError: name 'detect_spots' is not defined

# **2- Image Processing Functions**


- spot & Mold detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def detect_spots(
    image,
    mask,
    reference_lab=[80, 140, 110],
    threshold=30,
    reference_shift_threshold=25,
    defect_percentage_threshold=5
):

    # read image
    if isinstance(image, str):
        img_rgba = cv2.imread(
            image,
            cv2.IMREAD_UNCHANGED
        )
    else:
        img_rgba = np.array(image)

    if img_rgba is None:
        return "Image could not be loaded"

    if img_rgba.shape[2] == 4:

        img_rgb = img_rgba[:, :, :3]

        img_bgr = cv2.cvtColor(
            img_rgb,
            cv2.COLOR_RGB2BGR
        )

    else:

        img_bgr = cv2.cvtColor(
            img_rgba,
            cv2.COLOR_RGB2BGR
        )

    fruit_mask = mask > 0

    kernel = np.ones((15, 15), np.uint8)

    inner_mask = cv2.erode(
        fruit_mask.astype(np.uint8),
        kernel,
        iterations=1
    ).astype(bool)

    fruit_mask = inner_mask

    img_lab = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2LAB
    )

    fruit_pixels_lab = img_lab[fruit_mask]

    median_color = np.median(
        fruit_pixels_lab,
        axis=0
    )

    AB = img_lab[:, :, 1:3]

    fruit_AB = AB[fruit_mask]

    current_median_AB = np.percentile(
        fruit_AB,
        50,
        axis=0
    )

    reference_AB = np.array(
        reference_lab[1:3],
        dtype=np.float32
    )

    shift = np.linalg.norm(
        current_median_AB.astype(np.float32)
        - reference_AB
    )

    if shift > reference_shift_threshold:

        median_AB = reference_AB

    else:

        median_AB = current_median_AB

    diff = np.linalg.norm(
        AB.astype(np.float32)
        - median_AB,
        axis=2
    )

    defect_mask = (
        (diff > threshold)
        & fruit_mask
    )

    defect_ratio = (
        defect_mask.sum()
        / fruit_mask.sum()
    ) * 100

    if defect_ratio < defect_percentage_threshold:

        return "NO DEFECT"

    # visualization
    result = img_bgr.copy()

    result[defect_mask] = [0, 0, 0]

    result[~fruit_mask] = [255, 255, 255]

    rejected_pixels_bgr = img_bgr[defect_mask]

    if len(rejected_pixels_bgr) == 0:

        return "NO DEFECT"

    luminance = (
        0.114
        * rejected_pixels_bgr[:, 0].astype(np.float32)
        +
        0.587
        * rejected_pixels_bgr[:, 1].astype(np.float32)
        +
        0.299
        * rejected_pixels_bgr[:, 2].astype(np.float32)
    )

    closer_to_black = (
        luminance < 128
    ).sum()

    closer_to_white = (
        luminance >= 128
    ).sum()

    if closer_to_white > closer_to_black:

        majority_color = [255, 255, 255]

        majority_label = "MOLD"

    else:

        majority_color = [0, 0, 0]

        majority_label = "DARK SPOTS"


    result_voted = result.copy()

    result_voted[defect_mask] = majority_color

    return (f"{majority_label}, "f"{defect_ratio:.2f}%")

- Texture analysis

In [ ]:
# helpers
def get_mask(no_bg_image):
    mask = np.array(no_bg_image.split()[-1])
    return mask

In [ ]:
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

def texture_analysis(
    crop,
    mask,
    blur_kernel=5,
    canny_low=40,
    canny_high=120,
    wrinkle_kernel_size=9,
    shrivel_threshold=15,
    skin_texture_threshold=0.50
):
    if not isinstance(crop, np.ndarray):
        crop = np.array(crop.convert("RGB"))

    mask = (mask > 0).astype(np.uint8)

    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    gray = cv2.bitwise_and(gray, gray, mask=mask)
    blur = cv2.GaussianBlur(gray, (blur_kernel, blur_kernel), 0)

    # SHRIVEL
    edges = cv2.Canny(blur, canny_low, canny_high)
    edges = cv2.bitwise_and(edges, edges, mask=mask)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (wrinkle_kernel_size, wrinkle_kernel_size)
    )

    blackhat = cv2.morphologyEx(blur, cv2.MORPH_BLACKHAT, kernel)
    blackhat = cv2.bitwise_and(blackhat, blackhat, mask=mask)

    area = np.count_nonzero(mask)

    if area == 0:
        return {"is_shriveled": False, "has_texture_problem": False}

    edge_density = np.count_nonzero(edges) / area
    wrinkle_intensity = np.mean(blackhat[mask > 0])
    shrivel_score = (edge_density * 100) + wrinkle_intensity

    # TEXTURE
    fruit_pixels = blur[mask > 0]

    if len(fruit_pixels) == 0:
        return {"is_shriveled": False, "has_texture_problem": False}

    min_val = np.min(fruit_pixels)
    max_val = np.max(fruit_pixels)

    normalized = np.zeros_like(blur)

    if max_val > min_val:
        normalized[mask > 0] = (
            (blur[mask > 0] - min_val) / (max_val - min_val) * 255
        ).astype(np.uint8)

    glcm = graycomatrix(
        normalized,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(glcm, "contrast")[0, 0]
    homogeneity = graycoprops(glcm, "homogeneity")[0, 0]
    energy = graycoprops(glcm, "energy")[0, 0]

    lbp = local_binary_pattern(normalized, P=8, R=1, method="uniform")
    lbp = lbp.astype(np.uint8)
    lbp = cv2.bitwise_and(lbp, lbp, mask=mask)

    lbp_std = np.std(lbp[mask > 0])

    roughness = (contrast / 1000) + (lbp_std / 10)
    skin_texture_score = (homogeneity + energy) / (1 + roughness)

    return {
        "is_shriveled": bool(shrivel_score > shrivel_threshold),
        "has_texture_problem": bool(skin_texture_score < skin_texture_threshold or shrivel_score > shrivel_threshold)
    }



- Color Analysis — Expected Color Coverage

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def color_analysis(crop, mask,
                   ref_color_bgr,
                   delta_e_threshold,
                   min_coverage):

    # input handling for pipeline
    if not isinstance(crop, np.ndarray):
        crop = np.array(crop.convert("RGB"))

    # convert rgb to bgr, in order to then convert to lab
    crop = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
    mask = (mask > 0).astype(np.uint8)

    crop = cv2.medianBlur(crop, 5)
    lab  = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)

    # Reference colour converted to OpenCV LAB
    ref_lab = cv2.cvtColor(
        np.uint8([[ref_color_bgr]]), cv2.COLOR_BGR2LAB
    )[0][0].astype(np.float32)

    # Per-pixel Delta-E (Euclidean distance in LAB)
    delta_e = np.sqrt(
        (lab[:,:,0] - ref_lab[0])**2 +
        (lab[:,:,1] - ref_lab[1])**2 +
        (lab[:,:,2] - ref_lab[2])**2
    )

    # Pixels within threshold = healthy colour present
    color_mask = ((delta_e < delta_e_threshold) & (mask > 0)).astype(np.uint8) * 255

    # Morphological cleanup
    k = np.ones((5, 5), np.uint8)
    color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_OPEN,  k)
    color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_CLOSE, k)

    coverage         = np.count_nonzero(color_mask) / np.count_nonzero(mask)


    return {
    "uniform_color_percentage": float(coverage * 100),
    "passed": bool(coverage >= min_coverage)
    }

- Sprouting Detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def sprouting_detection(crop, mask):
    """
    Detects protrusions by comparing the actual mask
    to its convex hull. No color assumptions needed.
    """
    # convex hull for the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull_mask = np.zeros_like(mask)
    if contours:
        hull = cv2.convexHull(max(contours, key=cv2.contourArea))
        cv2.fillPoly(hull_mask, [hull], 255)

    # difference between the hull and the original mask
    protrusion_mask = cv2.bitwise_and(hull_mask, cv2.bitwise_not(mask))

    k = np.ones((3, 3), np.uint8)
    protrusion_mask = cv2.morphologyEx(protrusion_mask, cv2.MORPH_OPEN, k, iterations=2)

    # analyze protruding regions
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(protrusion_mask, 8)
    overlay = crop.copy()
    max_height = 0
    sprout_count = 0

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        aspect = max(w, h) / (min(w, h) + 1e-6)
        if area > 40 and aspect > 1.8:   # elongated protrusion
            max_height = max(max_height, max(w, h))
            sprout_count += 1
            cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 255, 0), 2)

    sprouted = sprout_count > 0 and max_height > 20

    show_row(
        ('Crop', crop),
        ('Convex Hull Gap', protrusion_mask, 'gray'),
        ('Sprout Boxes', overlay)
    )
    print(f'Sprouted: {sprouted}  |  Sprouts found: {sprout_count}  |  Max size: {max_height}px')
    return sprouted, max_height